In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle('../../data/processed/23_cleaned_master_data.pkl')

### Prüfung auf weitere Platzhalter neben NA Werte die wir bereits eliminiert haben

- wie in der untenstehenden Tabelle sehen kann sieht alles gut aus
- jediglich einige 0 Werte bei den Fahrer Profilen
- einige "-" bei won how

In [3]:
audit_data = []

for col in df.columns:
    col_series = df[col]
    col_str = col_series.astype(str).str.strip()

    # Echte NaNs & Unendliche Werte (inf)
    cnt_real_na = col_series.isna().sum()

    # inf-Check (nur für numerische Spalten sinnvoll)
    cnt_inf = 0
    if pd.api.types.is_numeric_dtype(col_series) and not pd.api.types.is_bool_dtype(col_series):
        cnt_inf = np.isinf(col_series).sum()

    # String 'nan' korrigieren (echte NaNs abziehen)
    cnt_string_nan = (col_str == 'nan').sum()
    if cnt_real_na > 0:
        cnt_string_nan = max(0, cnt_string_nan - cnt_real_na)

    # Numerische 0er
    if pd.api.types.is_bool_dtype(col_series):
        cnt_zero = 0
    else:
        cnt_zero = (col_series == 0).sum() + (col_str == '0').sum() if col_series.dtype != object else (col_str == '0').sum()


    col_str_lower = col_str.str.lower()

    entry = {
        'Spalte': col,
        'Echte_NaN': cnt_real_na,
        'Echte_inf': cnt_inf,
        'cnt_0': cnt_zero,
        'cnt_-': (col_str == '-').sum(),
        'cnt_?': (col_str == '?').sum(),
        'cnt_leer': (col_str == '').sum(),
        'txt_nan': cnt_string_nan,
        'txt_null': (col_str_lower == 'null').sum(),
        'txt_none': (col_str_lower == 'none').sum(),
        'txt_unknown': (col_str_lower == 'unknown').sum(),
        'txt_undefined': (col_str_lower == 'undefined').sum(),
        'txt_not_found': (col_str_lower == 'not found').sum() + (col_str_lower == 'notfound').sum(),
        'txt_tbd': (col_str_lower == 'tbd').sum()
    }
    audit_data.append(entry)

df_audit_extended = pd.DataFrame(audit_data)


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1400)

print(df_audit_extended)

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')

                         Spalte  Echte_NaN  Echte_inf  cnt_0  cnt_-  cnt_?  cnt_leer  txt_nan  txt_null  txt_none  txt_unknown  txt_undefined  txt_not_found  txt_tbd
0                     meta_race          0          0      0      0      0         0        0         0         0            0              0              0        0
1                     meta_year          0          0      0      0      0         0        0         0         0            0              0              0        0
2                      meta_url          0          0      0      0      0         0        0         0         0            0              0              0        0
3                          rank          0          0      0      0      0         0        0         0         0            0              0              0        0
4                meta_rider_url          0          0      0      0      0         0        0         0         0            0              0              0        0
5   